# Phase 12 - MoE-Diagnose: wie ist der Expertenblock wirklich gebaut?

**Braucht keine GPU und keine Gewichte** - nur die config (37 kB). Sekunden.

Die FFN-Zelle hat gemeldet, dass `model.layers.N.mlp.experts` **keine Kinder** hat.
Das ist die gebuendelte MoE-Umsetzung: die Expertengewichte liegen als gestapelte
Tensoren und werden mit einem Batch-Matmul gerechnet, nicht als 256 einzelne
Linear-Module. Haken auf `down_proj` gibt es dort nicht - daran ist der erste Anlauf
gescheitert.

Diese Zelle raet nicht, wie es stattdessen aussieht. Sie liest es aus: Parameternamen
und -formen, Modulstruktur, und vor allem den **Quelltext der forward-Methode**.
Damit steht fest, wie die FFN-Zwischenschicht berechnet wird und wo man sie abgreift.

Ist zufaellig noch ein Modell in der Sitzung geladen, wird es mitgenutzt (dann kommen
auch die echten Parameterformen dazu) - noetig ist es nicht.


In [ ]:
# === PHASE 12 - MoE-DIAGNOSE: WIE IST DER EXPERTENBLOCK WIRKLICH GEBAUT? ====
# Die FFN-Zelle hat gemeldet: 'model.layers.N.mlp.experts' hat KEINE Kinder.
# Das ist die gebuendelte Umsetzung - die Expertengewichte liegen als
# gestapelte Tensoren und werden mit einem Batch-Matmul gerechnet, nicht als
# 256 einzelne Linear-Module. Haken auf 'down_proj' gibt es dort nicht, und
# genau daran ist der erste Anlauf gescheitert.
#
# Diese Zelle raet nicht, wie es stattdessen aussieht. Sie liest es aus:
# Parameternamen und -formen des Expertenblocks, die Modulstruktur, und vor
# allem den QUELLTEXT der forward-Methode. Damit steht schwarz auf weiss, wie
# man die FFN-Zwischenschicht berechnet und wo man sie abgreift.
#
# BRAUCHT KEINE GPU und KEINE GEWICHTE. Nur die config (37 kB) - daraus wird
# die Modellklasse aufgeloest und ihr Modul importiert. Laeuft in einer
# CPU-Laufzeit in Sekunden. Ist zufaellig ein Modell in der Sitzung geladen,
# wird es genutzt, noetig ist es nicht.
import os, sys, json, inspect, importlib, re, time
import numpy as np
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_moe_diagnose")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
print("="*78)
print("MoE-DIAGNOSE fuer %s"%MODEL_ID)
print("="*78)
BLOCK=None; QUELLE=None
if "model" in globals():
    try:
        BLOCK=type(model.model.layers[0].mlp)
        print("Modell in der Sitzung gefunden - Klasse direkt entnommen.")
    except Exception as e:
        print("Modell da, aber mlp nicht erreichbar: %s"%e)
if BLOCK is None:
    from transformers import AutoConfig
    from transformers.models.auto.modeling_auto import MODEL_FOR_CAUSAL_LM_MAPPING
    cfg=AutoConfig.from_pretrained(MODEL_ID)
    inner=getattr(cfg,"text_config",cfg)
    kls=None
    for c in (type(cfg),type(inner)):
        if c in MODEL_FOR_CAUSAL_LM_MAPPING: kls=MODEL_FOR_CAUSAL_LM_MAPPING[c]; break
    assert kls is not None,"Modellklasse nicht aufloesbar - dann mit geladenem Modell laufen"
    mod=importlib.import_module(kls.__module__)
    print("Modul: %s"%kls.__module__)
    kandidaten=[(n,o) for n,o in vars(mod).items()
                if inspect.isclass(o) and re.search(r"(MoE|Moe|Sparse|Experts|MLP)",n)]
    print("Kandidaten-Klassen: %s"%", ".join(n for n,_ in kandidaten))
    for n,o in kandidaten:
        if re.search(r"(SparseMoe|MoeBlock|MoeSparse)",n): BLOCK=o; break
    if BLOCK is None and kandidaten: BLOCK=kandidaten[0][1]
print("")
print("EXPERTENBLOCK-KLASSE: %s"%getattr(BLOCK,"__name__","?"))
# ---- Parameter und Module, wenn ein echtes Modell da ist -------------------
if "model" in globals():
    m0=model.model.layers[0].mlp
    print("")
    print("PARAMETER unter model.layers.0.mlp (Name, Form, dtype):")
    for n,p in m0.named_parameters():
        print("  %-42s %-24s %s"%(n,tuple(p.shape),str(p.dtype).replace("torch.","")))
    print("")
    print("MODULE unter model.layers.0.mlp:")
    for n,mm in m0.named_modules():
        if n: print("  %-42s %s"%(n,type(mm).__name__))
    print("")
    print("PUFFER (nicht-trainierbare Tensoren):")
    for n,b in m0.named_buffers(): print("  %-42s %s"%(n,tuple(b.shape)))
else:
    print("")
    print("(kein Modell geladen - Parameterformen entfallen, Quelltext kommt trotzdem)")
# ---- der eigentliche Zweck: der Quelltext ---------------------------------
for kls,titel in ((BLOCK,"forward des Expertenblocks"),
                  (getattr(BLOCK,"__mro__",[None])[0],None)):
    if kls is None or titel is None: continue
    print("")
    print("="*78); print(titel.upper()); print("="*78)
    try:
        print(inspect.getsource(kls.forward))
    except Exception as e:
        print("  Quelltext nicht lesbar: %s"%e)
EXP=None
if "model" in globals():
    EXP=type(model.model.layers[0].mlp.experts)
elif BLOCK is not None:
    mod=importlib.import_module(BLOCK.__module__)
    for n,o in vars(mod).items():
        if inspect.isclass(o) and n.endswith("Experts"): EXP=o; break
if EXP is not None:
    print("")
    print("="*78); print("FORWARD DER EXPERTEN-SAMMLUNG: %s"%EXP.__name__); print("="*78)
    try: print(inspect.getsource(EXP.forward))
    except Exception as e: print("  nicht lesbar: %s"%e)
    print("")
    print("__init__ der Experten-Sammlung (zeigt die Gewichtsform):")
    try: print(inspect.getsource(EXP.__init__))
    except Exception as e: print("  nicht lesbar: %s"%e)
print("")
print("="*78)
print("WAS ICH DARAUS BRAUCHE")
print("="*78)
print("1. wie die FFN-Zwischenschicht heisst und berechnet wird")
print("   (gate_up gestapelt? getrennt? welche Aktivierung? welche Achse ist")
print("    der Experte, welche die Zwischenbreite?)")
print("2. an welchem Tensor man sie abgreifen kann, ohne den Vorwaertspass")
print("   nachzubauen - oder ob man sie aus dem Eingang des Blocks und den")
print("   gestapelten Gewichten selbst rechnen muss")
print("3. wie die Router-Auswahl (top-8) zurueckgegeben wird")
print("")
print("Damit schreibe ich die FFN-Zelle gegen den tatsaechlichen Aufbau statt")
print("gegen einen vermuteten. Bitte die Ausgabe komplett zurueckschicken.")
